# UniLab Demo: One-command checkpoint playback and teaser rendering

`uv run demo <name>` is the simplest interactive entrypoint in UniLab:

- On first run, it automatically fetches checkpoint weights for the checkpoint demos or scene assets for `teaser` from Hugging Face.
- After the assets are cached locally, it launches the interactive simulator or scene renderer.
- No training run or manually prepared checkpoint is required.

## Available demos

| Name | Type | Description |
| --- | --- | --- |
| `dance` | checkpoint | G1 whole-body motion tracking with a dance motion |
| `wallflip` | checkpoint | G1 wall flip |
| `boxtracking` | checkpoint | G1 box-pushing tracking |
| `locomani` | checkpoint | Go2 with arm manipulation-locomotion |
| `inhandgrasp` | checkpoint | Sharpa in-hand rotation grasping |
| `teaser` | renderer | Static MotrixSim teaser scene from the paper |

## Environment setup

If you already completed the Quick Demo in the README (`make setup-motrix` or `uv sync --extra motrix`), you can skip this section.

> **Local execution only**: every demo opens a local GUI window (MotrixSim viewer or MuJoCo viewer). Google Colab is not supported.

## Step 1: Check the environment

Make sure this notebook is running from the UniLab project root.

In [ ]:
from pathlib import Path

root = Path(".").resolve()
assert (root / "scripts").is_dir(), (
    f"Run this notebook from the UniLab project root. Current directory: {root}"
)
assert (root / "conf").is_dir(), (
    "Missing conf/ directory. Please check that the repository checkout is complete."
)
print(f"Project root: {root}")

## Step 2: Inspect the demo registry and local cache status

Checkpoint caches live under `src/unilab/assets/checkpoints/<demo-name>/model_0.pt`. The first `uv run demo <name>` run fetches them from the Hugging Face dataset [`unilabsim/unilab-checkpoints`](https://huggingface.co/datasets/unilabsim/unilab-checkpoints). `teaser` uses the separate [`unilabsim/unilab-scenes`](https://huggingface.co/datasets/unilabsim/unilab-scenes) dataset and is handled internally by `unilab.tools.render_teaser`.

In [ ]:
from unilab.assets import ASSETS_ROOT_PATH
from unilab.demo import DEMO_REGISTRY

cache_dir = ASSETS_ROOT_PATH / "checkpoints"
for name in sorted(DEMO_REGISTRY):
    spec = DEMO_REGISTRY[name]
    if spec.entry == "teaser":
        status = "scene assets; fetched from unilab-scenes on first run"
    else:
        pt = cache_dir / name / "model_0.pt"
        status = "cached locally" if pt.is_file() else "not cached; downloaded on first run"
    print(f"  {name:12s}  entry={spec.entry:18s}  {status}")

## Step 3: Run a demo

The cell below shows how to launch `dance`. Each demo opens an **interactive simulator window**: four Motrix checkpoint demos use the MotrixSim viewer, `locomani` uses the MuJoCo viewer, and `teaser` renders a static MotrixSim scene. When run from the notebook, the cell blocks until you close the window.

To try another demo, replace `dance` with `wallflip`, `boxtracking`, `locomani`, `inhandgrasp`, or `teaser`.

In [ ]:
!uv run demo dance

## Options

- `--refresh` - force checkpoint re-download by deleting the local cache before resolving it. `teaser` ignores this option.
- `--device cpu` / `--device cuda:0` - explicitly choose the inference device. This only applies to checkpoint demos.

```bash
uv run demo dance --refresh
uv run demo wallflip --device cpu
```

## Tab completion

Running `make setup-motrix` registers the completion script in your shell rc file. After opening a new shell:

```text
uv run demo <TAB>            -> boxtracking dance inhandgrasp locomani teaser wallflip
uv run demo d<TAB>           -> dance
uv run demo dance --<TAB>    -> --device --refresh
```

## Next steps

- Train your own checkpoint: `uv run train --algo ppo --task <task> --sim motrix`
- Inspect the dispatch logic: `DEMO_REGISTRY` and `build_demo_command` in [src/unilab/demo.py](../src/unilab/demo.py)